# Baseline Submission
Generate submission.csv from the original pipeline (no boost improvements).  
Retrains pointwise + lambdarank on 100% training data, predicts on test.csv.

In [1]:
import sys
from pathlib import Path

cwd = Path(".").resolve()
REPO_ROOT = cwd if (cwd / ".git").exists() else cwd.parent
sys.path.insert(0, str(REPO_ROOT))

import json
import numpy as np
import pandas as pd
import lightgbm as lgb

from src.utils import load_train, load_test, make_submission
from src.features import (
    build_features,
    add_relative_features,
    add_price_features,
    add_competitor_features,
    add_missing_indicators,
    add_temporal_features,
)
from src.svd_features import build_svd_features
from src.position_features import build_position_features
from src.modeling import get_lgb_device_params

DEVICE_PARAMS = get_lgb_device_params()
OUT_DIR = REPO_ROOT / "outputs"
print(f"REPO_ROOT: {REPO_ROOT}")

GPU mode enabled - using LightGBM GPU acceleration
REPO_ROOT: /home/akshitanchan/Projects/hotel-trivago-but-its-expedia


## Load and Prepare Training Data

In [2]:
print("Loading full training data...")
train_df = load_train()
print(f"Train: {len(train_df):,} rows, {train_df['srch_id'].nunique():,} searches")

train_df = build_features(train_df)

# SVD and position features: use ALL training data as the "fold"
# (no holdout — we're training on everything)
train_df = build_svd_features(train_df, train_df)
train_df = build_position_features(train_df, train_df)

print(f"Features built: {train_df.shape[1]} columns")

Loading full training data...
Train: 4,958,347 rows, 199,795 searches


/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["comp_cheaper_frac"] = (
/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:104: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_missing"] = df[col].isnull().astype("int8")
/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:104: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

Features built: 167 columns


## Prepare Feature Columns

In [3]:
DROP_COLS = [
    "srch_id", "date_time", "position", "click_bool", "booking_bool",
    "gross_bookings_usd", "random_bool",
]
COMP_RAW = [f"comp{i}_{s}" for i in range(1, 9) for s in ("rate", "inv", "rate_percent_diff")]
drop = set(DROP_COLS + COMP_RAW)
feature_cols = [c for c in train_df.columns if c not in drop]
print(f"Training with {len(feature_cols)} features")

Training with 136 features


## Train Pointwise Model (100% data, fixed rounds)

Exact params from `outputs/pointwise_run1.json`: leaves=255, lr=0.01, min_child=100.  
num_boost_round=3998 (original best_iteration). No early stopping — no val set.

In [4]:
pw_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "num_leaves": 255,
    "learning_rate": 0.01,
    "min_child_samples": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "feature_pre_filter": False,
    "verbose": -1,
    "seed": 42,
    **DEVICE_PARAMS,
}

y_train_pw = train_df["booking_bool"].values
X_train = train_df[feature_cols]

print("Training pointwise model (3998 rounds, no early stopping)...")
pw_dataset = lgb.Dataset(X_train, label=y_train_pw)
pw_model = lgb.train(
    pw_params,
    pw_dataset,
    num_boost_round=3998,
    callbacks=[lgb.log_evaluation(500)],
)
print("Pointwise training complete.")

Training pointwise model (3998 rounds, no early stopping)...
Pointwise training complete.


## Train LambdaRank Model (100% data, fixed rounds)

Exact params from `outputs/lambdarank_run1.json`: same leaves/lr/min_child.  
num_boost_round=947. No early stopping.

In [5]:
lr_params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [5],
    "num_leaves": 255,
    "learning_rate": 0.01,
    "min_child_samples": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "feature_pre_filter": False,
    "verbose": -1,
    "seed": 42,
    **DEVICE_PARAMS,
}

y_train_rel = (
    train_df["booking_bool"] * 5
    + (train_df["click_bool"] - train_df["booking_bool"]).clip(lower=0)
).values
group_train = train_df.groupby("srch_id").size().values

print("Training lambdarank model (947 rounds, no early stopping)...")
lr_dataset = lgb.Dataset(X_train, label=y_train_rel, group=group_train)
lr_model = lgb.train(
    lr_params,
    lr_dataset,
    num_boost_round=947,
    callbacks=[lgb.log_evaluation(200)],
)
print("LambdaRank training complete.")

Training lambdarank model (947 rounds, no early stopping)...
LambdaRank training complete.


## Load and Prepare Test Data

SVD and position feature maps are computed from training data and applied to test.

In [6]:
print("Loading test data...")
test_df = load_test()
print(f"Test: {len(test_df):,} rows, {test_df['srch_id'].nunique():,} searches")

# Build features on test data
test_df = build_features(test_df)

# Apply SVD and position features using training-derived maps
print("Applying SVD features from training data to test...")
test_df = build_svd_features(train_df, test_df)
print("Applying position-debiased features from training data to test...")
test_df = build_position_features(train_df, test_df)

# Verify feature alignment
test_features = [c for c in test_df.columns if c in feature_cols]
missing = set(feature_cols) - set(test_features)
if missing:
    print(f"WARNING: {len(missing)} features missing in test: {missing}")
    for col in missing:
        test_df[col] = 0  # fill missing features with 0
print(f"Test features: {len(test_features)}/{len(feature_cols)} present")
X_test = test_df[feature_cols]

# Check for NaN
nan_counts = X_test.isnull().sum()
nan_cols = nan_counts[nan_counts > 0]
if len(nan_cols) > 0:
    print(f"NaN in {len(nan_cols)} test features:")
    for col, cnt in nan_cols.items():
        pct = cnt / len(X_test) * 100
        print(f"  {col}: {cnt:,} ({pct:.1f}%)")
else:
    print("No NaN values in test features.")

Loading test data...
Test: 4,959,183 rows, 199,549 searches


/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:104: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_missing"] = df[col].isnull().astype("int8")
/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:104: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_missing"] = df[col].isnull().astype("int8")
/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:104: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fra

Applying SVD features from training data to test...
Applying position-debiased features from training data to test...
Test features: 136/136 present
NaN in 22 test features:
  visitor_hist_starrating: 4,705,752 (94.9%)
  visitor_hist_adr_usd: 4,704,559 (94.9%)
  prop_review_score: 7,266 (0.1%)
  prop_location_score2: 1,088,032 (21.9%)
  srch_query_affinity_score: 4,641,025 (93.6%)
  orig_destination_distance: 1,608,679 (32.4%)
  prop_review_score_rank: 7,266 (0.1%)
  prop_review_score_diff_mean: 7,266 (0.1%)
  prop_review_score_diff_median: 7,266 (0.1%)
  prop_review_score_min: 8 (0.0%)
  prop_review_score_max: 8 (0.0%)
  prop_review_score_std: 8 (0.0%)
  prop_location_score2_rank: 1,088,032 (21.9%)
  prop_location_score2_diff_mean: 1,088,032 (21.9%)
  prop_location_score2_diff_median: 1,088,032 (21.9%)
  prop_location_score2_min: 91,461 (1.8%)
  prop_location_score2_max: 91,461 (1.8%)
  prop_location_score2_std: 186,298 (3.8%)
  price_vs_visitor_hist: 4,704,559 (94.9%)
  star_diff_vis

## Predict and Blend

Min-max normalize per search, then 50/50 blend (original ensemble weights).

In [7]:
print("Generating predictions...")
test_df["pw_score"] = pw_model.predict(X_test)
test_df["lr_score"] = lr_model.predict(X_test)


def normalize_per_search(df, col):
    """Min-max normalize scores within each search."""
    g = df.groupby("srch_id")[col]
    mn, mx = g.transform("min"), g.transform("max")
    rng = (mx - mn).replace(0, 1)
    return ((df[col] - mn) / rng).astype("float32")


test_df["pw_norm"] = normalize_per_search(test_df, "pw_score")
test_df["lr_norm"] = normalize_per_search(test_df, "lr_score")

# 50/50 blend (original ensemble weights)
alpha = 0.5
test_df["final_score"] = alpha * test_df["lr_norm"] + (1 - alpha) * test_df["pw_norm"]

print("Predictions generated.")
print(f"  pw_score range: [{test_df['pw_score'].min():.4f}, {test_df['pw_score'].max():.4f}]")
print(f"  lr_score range: [{test_df['lr_score'].min():.4f}, {test_df['lr_score'].max():.4f}]")

Generating predictions...
Predictions generated.
  pw_score range: [0.0000, 0.9690]
  lr_score range: [-4.2193, 3.8689]


## Generate Submission + Sanity Checks

In [8]:
sub = make_submission(test_df, "final_score", REPO_ROOT / "submission_baseline.csv")

# Sanity checks
print(f"\n=== Sanity Checks ===")
print(f"Total rows: {len(sub):,} (expected: {len(test_df):,})")
assert len(sub) == len(test_df), "Row count mismatch!"

print(f"Duplicates: {sub.duplicated().sum()}")
assert sub.duplicated().sum() == 0, "Duplicate rows found!"

test_searches = set(test_df["srch_id"])
sub_searches = set(sub["SearchId"])
print(f"Searches in test: {len(test_searches):,}")
print(f"Searches in submission: {len(sub_searches):,}")
assert test_searches == sub_searches, "Search ID mismatch!"

# Spot-check a few searches
print(f"\nSpot check - first 3 searches:")
for srch_id in sorted(test_searches)[:3]:
    top5 = sub[sub["SearchId"] == srch_id].head(5)
    print(f"  Search {srch_id}: top props = {top5['PropertyId'].tolist()}")

print(f"\nSubmission saved to: {REPO_ROOT / 'submission_baseline.csv'}")
print(f"Header: {','.join(sub.columns)}")
print(f"Ready to upload to Kaggle!")


=== Sanity Checks ===
Total rows: 4,959,183 (expected: 4,959,183)
Duplicates: 0
Searches in test: 199,549
Searches in submission: 199,549

Spot check - first 3 searches:
  Search 1: top props = [54937, 28181, 99484, 61934, 90385]
  Search 3: top props = [131940, 63039, 79176, 47377, 46861]
  Search 6: top props = [31470, 24695, 5333, 83424, 103320]

Submission saved to: /home/akshitanchan/Projects/hotel-trivago-but-its-expedia/submission_baseline.csv
Header: SearchId,PropertyId
Ready to upload to Kaggle!
